In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/src/task4_enriched_profit

In [0]:
import pytest
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date
from decimal import Decimal

In [0]:
# mimic profit_by_year table creation (without calling main_enrich)
enriched_orders = spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
profit_df = build_profit_by_year(enriched_orders)
profit_df = dedupe(profit_df)
delta_writer(profit_df, "az_ci_de_dbs.ecom_dev.test_profit_by_year")

# Mimic table 
test_profit_by_year_df = spark.read.table("az_ci_de_dbs.ecom_dev.test_profit_by_year")

# Test build_profit_by_year output

def test_build_profit_by_year_not_empty():
    result = test_profit_by_year_df
    assert result.count() > 0, "build_profit_by_year returned empty DataFrame"


def test_build_profit_by_year_expected_columns():
    result = test_profit_by_year_df
    expected_cols = ["Year", "product_category", "product_sub_category", "customer", "total_profit"]
    assert sorted(result.columns) == sorted(expected_cols), f"Expected {sorted(expected_cols)}, got {sorted(result.columns)}"


def test_build_profit_by_year_no_extra_columns():
    result = test_profit_by_year_df
    assert len(result.columns) == 5, f"Expected 5 columns, got {len(result.columns)}"


def test_build_profit_by_year_int_year():
    result = test_profit_by_year_df
    year_type = str(result.schema["Year"].dataType)
    assert "int" in year_type.lower(), f"Year column should be integer type, got {year_type}"

def test_build_profit_by_year_no_null_groupby_cols():
    result = test_profit_by_year_df
    for col_name in ["Year", "product_category", "product_sub_category", "customer"]:
        nulls = result.filter(F.col(col_name).isNull()).count()
        assert nulls == 0, f"Found {nulls} nulls in {col_name}"


def test_build_profit_by_year_no_null_total_profit():
    result = test_profit_by_year_df
    nulls = result.filter(F.col("total_profit").isNull()).count()
    assert nulls == 0, f"Found {nulls} nulls in total_profit"


def test_build_profit_by_year_profit_rounded_to_2_places():
    result = test_profit_by_year_df
    violations = result.filter(
        F.col("total_profit") != F.round(F.col("total_profit"), 2)
    ).count()
    assert violations == 0, f"Found {violations} rows with profit not rounded to 2 places"


def test_build_profit_by_year_allows_negative_profit():
    result = test_profit_by_year_df
    neg_count = result.filter(F.col("total_profit") < 0).count()
    assert neg_count >= 0, "Negative profit check failed"


def test_build_profit_by_year_no_duplicates():
    result = test_profit_by_year_df
    total = result.count()
    distinct = result.select("Year", "product_category", "product_sub_category", "customer").distinct().count()
    assert total == distinct, f"Duplicate groups found: {total} rows, {distinct} distinct groups"


# Test profit_by_year table

def test_profit_by_year_table_exists():
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.test_profit_by_year"), "profit_by_year table does not exist"


def test_profit_by_year_table_not_empty():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.test_profit_by_year")
    assert df.count() > 0, "profit_by_year table is empty"


def test_profit_by_year_table_no_duplicates():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.test_profit_by_year")
    total = df.count()
    distinct = df.dropDuplicates().count()
    assert total == distinct, f"Duplicates in profit_by_year: {total} rows, {distinct} distinct"


def test_profit_by_year_table_matches_build_output():
    table_count = spark.read.table("az_ci_de_dbs.ecom_dev.test_profit_by_year").count()
    build_count = dedupe(build_profit_by_year(enriched_orders)).count()
    assert table_count == build_count, f"Table count ({table_count}) != build output ({build_count})"


# --- Run all tests ---

def main_enriched_profit_test():
    test_functions = [
        
        # build_profit_by_year
        test_build_profit_by_year_not_empty,
        test_build_profit_by_year_expected_columns,
        test_build_profit_by_year_no_extra_columns,
        test_build_profit_by_year_no_null_groupby_cols,
        test_build_profit_by_year_no_null_total_profit,
        test_build_profit_by_year_profit_rounded_to_2_places,
        test_build_profit_by_year_allows_negative_profit,
        test_build_profit_by_year_no_duplicates,
        # profit_by_year table
        test_profit_by_year_table_exists,
        test_profit_by_year_table_not_empty,
        test_profit_by_year_table_no_duplicates,
        test_profit_by_year_table_matches_build_output,
    ]

    passed, failed = 0, 0
    for fn in test_functions:
        try:
            fn()
            passed += 1
            print(f"  [PASSED] {fn.__name__}")
        except Exception as e:
            failed += 1
            print(f"  [FAILED] {fn.__name__}: {e}")

    print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")

    # Cleanup: drop mimicked table
    spark.sql("DROP TABLE IF EXISTS az_ci_de_dbs.ecom_dev.profit_by_year")
    print("\n[CLEANUP] Dropped az_ci_de_dbs.ecom_dev.test_profit_by_year")